# First α analysis from Lee et al. 2026 (PRL 136, 033801), Fig. 3b

Digitized drift-rate histories of four cryogenic Si cavities, plus the honest constant-power analysis.

**Provenance.** arXiv:2509.13503v1 ships three figure images and **no** machine-readable supplement
(checked the source tarball), so Fig. 3b (`drift_07102025.jpg`) was digitized by
`scripts/digitize_fig3b.py`: color segmentation against hand-calibrated broken-log axes,
log-binned medians, ~15% per-point error. Two corrections to the brief that motivated this:
(1) there is no CSV in the anc files — the tarball has figures only;
(2) Si6 runs at **17 K** and Si2/Si3/Si5 at **124 K** (zero-crossings), not 4 K — the protocol's
β≈0 argument holds at the 124 K crossing with ~1e-18/s aging, and is cleanest toward 4 K-class operation.

**Honesty note.** With power held constant, α·P is degenerate with β_Si: no α fit is possible.
This notebook reports (a) drift levels and decay shapes, (b) the Si2 cumulative-integral
cross-check (-44 kHz / 10 yr), and (c) sensitivity projections showing what logged
power excursions would buy. The full v3.1 regression needs P_trans(t) + T(t) from the labs.

In [ ]:
import matplotlib
try:
    get_ipython  # noqa: F821 - only defined inside Jupyter
except NameError:
    matplotlib.use('Agg')
import matplotlib.pyplot as plt
import runpy
from pathlib import Path
from vacuum_creep import literature as L

print('nu0 = %.4e Hz (1542 nm)' % L.NU0_HZ)
for cav, a in L.PAPER_ANCHORS.items():
    s = a['recent_uHz_s'] * 1e-6
    print(f"{cav}: {a['recent_uHz_s']:+.0f} uHz/s -> {L.hz_per_s_to_frac(s):+.2e}/s (paper {a['recent_frac_s']:+.1e})")
print('Si2 -44 kHz/10yr -> lengthening %.1f pm (paper 48 pm)' % (L.lengthening_m(-44e3, 0.21) * 1e12))
print('Si6 P_circ = %.2f mW (90 nW x 470000/pi)' % (L.SI6_P_CIRC_W * 1e3))

In [ ]:
DATA = Path(L.DATA_PATH)
if not L.has_digitized_data(DATA):
    print('digitized CSV missing; running scripts/digitize_fig3b.py ...')
    runpy.run_path('scripts/digitize_fig3b.py', run_name='__main__')
df = L.load_digitized(DATA)
print(df.groupby('cavity').size())
fig, ax = plt.subplots(figsize=(8, 5))
for cav, g in df.groupby('cavity'):
    ax.plot(g['days_since_contacting'], g['drift_rate_uHz_s'].abs(), '.', ms=3, label=cav)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('days since optical contacting')
ax.set_ylabel('|drift rate| (uHz/s)')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.savefig('lee2026_fig3b_redigitized.png', dpi=100)
print('saved lee2026_fig3b_redigitized.png')

In [ ]:
print(f"{'cav':4s} {'span/d':>13s} {'integral/kHz':>13s} | settled median+-std (uHz/s)")
era = {'Si2': 2500, 'Si3': 2500, 'Si5': 2000, 'Si6': 400}
for cav in ('Si2', 'Si3', 'Si5', 'Si6'):
    sub = df[df.cavity == cav]
    tot = L.integrate_drift(df, cav)
    med, std, n = L.settled_stats(df, cav, days_min=era[cav])
    print(f'{cav:4s} {sub.days_since_contacting.min():5.0f}-{sub.days_since_contacting.max():5.0f} '
          f'{tot / 1e3:+8.1f} | {med:+7.1f}+-{std:4.1f} (n={n})')
print('paper: Si2 -44 kHz / 10 yr (digitized span starts day 260, so expect slightly less)')

In [ ]:
print('Constant-power sensitivity: |alpha| <= scatter/deltaP; dn/yr per 100 uW (circ).')
print('Using Si6 settled scatter and Si6 P_circ = %.1f mW:' % (L.SI6_P_CIRC_W * 1e3))
med, std, n = L.settled_stats(df, 'Si6', 400)
sig = abs(std) * 1e-6
for frac in (0.01, 0.10, 1.00):
    dP = frac * L.SI6_P_CIRC_W
    a, dn = L.constant_power_sensitivity(sig, dP)
    print(f'  excursion {frac * 100:5.1f}% (dP={dP * 1e3:6.2f} mW): |alpha|<={a:.2e} Hz/s/W  dn/yr<={dn:.2e}')
print('Takeaway: without logged power excursions there is no alpha bound yet;')
print('request P_trans(t) + T_cryo(t) from PTB/JILA and run the full v3.1 regression.')

In [ ]:
# Option (a): windowed beta for Si2 — early vs late halves.
# Si2 contacting ~mid-2015 (its 3798-day span ends late-2025), so day 2000 ~= Jan 2021:
# early ~= 2015-2020 (fast aging), late ~= 2021-2026 (settled).
SPLIT_DAYS = 2000.0
w = L.windowed_stats(df, 'Si2', SPLIT_DAYS)
print('Si2 beta-window split at day %.0f (~Jan 2021):' % SPLIT_DAYS)
for key in ('full', 'early', 'late'):
    med, std, n = w[key]
    print(f'  {key:5s}: {med:+7.1f}+-{std:6.1f} uHz/s (n={n})')
# Illustrative conditional bound at a 10% excursion of Si6-scale 13.5 mW
# circulating power (Si2 power unpublished; same ruler shows the gain).
DP = 0.10 * L.SI6_P_CIRC_W
for key in ('full', 'early', 'late'):
    _, std, _ = w[key]
    a, dn = L.constant_power_sensitivity(abs(std) * 1e-6, DP)
    print(f'  {key:5s}: |alpha|<={a:.2e} Hz/s/W  dn/yr<={dn:.2e} per 100 uW')
print('Late-window bound tightens %.1fx vs full-span (aging bias removed).'
      % (w['full'][1] / w['late'][1]))

In [ ]:
# Option (b): parametric aging — exponential vs 1/t on Si2 (days >= 800,
# past the red-overlap zone where early black pixels are compromised).
SUB0 = 800.0
b0t, b1t, rst, nt = L.aging_fit_1overT(df, 'Si2', SUB0)
b0e, b1e, taue, rse, ne = L.aging_fit_exp(df, 'Si2', SUB0)
print('Si2 aging fits, days>=%.0f (n=%d):' % (SUB0, ne))
print('  1/t: beta0=%+.1f resid=%.1f uHz/s' % (b0t, rst))
print('  exp: beta0=%+.1f tau=%.0fd (%.2f yr) resid=%.1f uHz/s' % (b0e, taue, taue / 365.25, rse))
print('Exp asymptote matches settled -57; 1/t asymptote (-4) does not.')
DP = 0.10 * L.SI6_P_CIRC_W
for tag, sd in (('windowed-late', L.windowed_stats(df, 'Si2', 2000.0)['late'][1]), ('exp-resid', rse)):
    a, dn = L.constant_power_sensitivity(abs(sd) * 1e-6, DP)
    print(f'  {tag}: dn/yr<={dn:.2e} per 100 uW (same 10% ruler)')
tot = L.integrate_drift(df, 'Si2')
early_mean = (-44e3 - tot) / (260 * 86400) * 1e6
print('Span integral %.1f kHz; missing days 0-260 imply early mean %.0f uHz/s (cf Si3 -883 at day 269).'
      % (tot / 1e3, early_mean))

In [ ]:
# Dither reach from first principles: Si6 13.5 -> 27 mW alternating months.
DP = L.SI6_P_CIRC_W
floor = L.slope_precision_allan()
print('Allan statistical floor, 1-month slope: %.2e Hz/s -> negligible.' % floor)
print('Reach is set by per-month repeatability sigma_m (beta wander + thermal):')
for sm in (1.0, 3.0, 10.0):
    _, dn2 = L.dither_sensitivity(sm, DP, 1)
    _, dn12 = L.dither_sensitivity(sm, DP, 6)
    print(f'  sigma_m={sm:4.1f} uHz/s: 2 months dn<={dn2:.2e} / 1 yr alternating dn<={dn12:.2e}')
print('Draft numbers 1.2e-15 / 3e-16 need sigma_m ~= 0.7 / 0.4 uHz/s: optimistic; use table.')

## Data request (send to PTB / JILA corresponding authors)

> Subject: Request for Si-cavity auxiliary logs for vacuum-index bound (Lee et al. 2026 follow-up)
>
> Dear colleagues — We are testing whether long-term cryogenic-silicon drift records bound
> power-dependent vacuum-index creep ("grooving"), per Protocol v3.1
> (s_i = β_Si + α·⟨P⟩_i + γ·ΔT_i on 10-day epochs, the same binning as your Fig. 3b).
> The published drift curves alone cannot separate α from β_Si at constant power.
> Could you share, at 10-day means matching your drift-rate bins:
> (1) transmitted power P_trans(t) (Si2/Si3/Si5/Si6),
> (2) cryostat temperature T(t),
> (3) lock/relock flags?
> We need only means, not Hz data, and will cite the PRL plus a data DOI of your choice.

## Next step after the logs arrive

Feed 10-day (s_i, ⟨P⟩_i, ΔT_i) rows per cavity into `vacuum_creep.run_pipeline`
(or the epoch regression directly) — that is the full Protocol v3.1 bound, no new apparatus.